# 2.6 — Optimistic Initial Values

---

## Initial values are a hidden input

Everything so far has quietly depended on $Q_1(a)$. For sample-average methods the
dependence vanishes after each action is tried once (recall $\alpha_1 = 1/1 = 1$
overwrites $Q_1$ completely). For constant-$\alpha$ methods it is **permanent**,
decaying only as $(1-\alpha)^n$.

Usually this is called *bias* and treated as a nuisance. It can also be used as a tool.

## The trick

Set $Q_1(a) = +5$ for all $a$, when the true values are $\mathcal{N}(0,1)$ — so every
arm looks wildly better than it is. Now run **pure greedy** ($\varepsilon = 0$):

1. Greedy picks some arm. Its reward is around 0, far below 5.
2. The update pulls that arm's estimate *down*, below the other arms' untouched 5.
3. Greedy now prefers a different arm. Repeat.

The agent is "disappointed" into systematically trying everything. Exploration emerges
from pure exploitation of a deliberately wrong prior. **No $\varepsilon$ required.**

$$\text{optimism} \;\Rightarrow\; \text{every untried action looks best} \;\Rightarrow\; \text{coverage}$$

## Its limits (the important part)

- **It is a transient.** Once the estimates come down to reality, the drive to explore is
  gone permanently. It is a *starting* condition, not an ongoing strategy.
- **It fails on nonstationary problems**, where you need exploration *forever*. The
  drive is triggered by the beginning of time, and the beginning of time only happens once.
- **It needs to know the scale of the rewards.** "+5" is only optimistic if you know
  rewards are around 0. In a new problem you often do not.

Sutton & Barto are explicit that this is a limited trick — but it isolates a real idea
(directed exploration via uncertainty) that UCB in 2.7 makes principled.

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

import numpy as np
import matplotlib.pyplot as plt
from bandit_utils import (Testbed, run_bandit, plot_pair, argmax_random_tiebreak,
                          hide, banner, RUNS, STEPS)

plt.rcParams["figure.dpi"] = 110
banner()

In [ ]:
class EpsGreedy:
    def __init__(self, k, runs, rng, eps=0.1, Q_init=0.0, alpha=None):
        self.k, self.runs, self.rng, self.eps = k, runs, rng, eps
        self.alpha = alpha
        self.Q = np.full((runs, k), float(Q_init))
        self.N = np.zeros((runs, k))

    def act(self):
        greedy = argmax_random_tiebreak(self.Q, self.rng)
        rand = self.rng.integers(0, self.k, size=self.runs)
        explore = self.rng.random(self.runs) < self.eps
        return np.where(explore, rand, greedy)

    def update(self, a, r):
        idx = np.arange(self.runs)
        self.N[idx, a] += 1
        step = self.alpha if self.alpha else 1.0 / self.N[idx, a]
        self.Q[idx, a] += step * (r - self.Q[idx, a])


def eps_agent(eps, Q_init=0.0, alpha=None):
    return lambda k, runs, rng: EpsGreedy(k, runs, rng, eps=eps,
                                          Q_init=Q_init, alpha=alpha)

print("agent defined")

---

### Predict first

We will compare, on the standard stationary testbed:

- **A:** optimistic greedy, $Q_1 = 5$, $\varepsilon = 0$, $\alpha = 0.1$
- **B:** realistic $\varepsilon$-greedy, $Q_1 = 0$, $\varepsilon = 0.1$, $\alpha = 0.1$

1. Which is better over the **first ~20 steps**?
2. Which is better at step 1000?
3. Predict the *shape* of A's % optimal curve in the first 30 steps. Will it be smooth?

*Write your guess down (mentally or in the cell below) before running the next cell. The point is not to be right — it is to make the surprise informative when you are wrong.*

In [ ]:
opt = run_bandit(eps_agent(0.0, Q_init=5.0, alpha=0.1), seed=2)
real = run_bandit(eps_agent(0.1, Q_init=0.0, alpha=0.1), seed=2)

fig, axes = plot_pair([opt, real],
                      ["optimistic greedy  $Q_1{=}5,\\ \\varepsilon{=}0$",
                       "realistic  $Q_1{=}0,\\ \\varepsilon{=}0.1$"],
                      title="Figure 2.3")
plt.show()

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(opt["optimal"][:60], "o-", ms=3, label="optimistic greedy")
ax.plot(real["optimal"][:60], "o-", ms=3, label="realistic $\\varepsilon$-greedy")
ax.axvline(10, color="gray", ls="--", lw=1)
ax.text(10.5, 5, "step 11", fontsize=8, color="gray")
ax.set_xlabel("Step"); ax.set_ylabel("% optimal"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title("Zoom on the first 60 steps - note the spike")
plt.show()

print("optimistic %opt at steps 8-14:", np.round(opt['optimal'][7:14], 1))

In [ ]:
hide('''<b>Early:</b> optimistic greedy is <i>worse</i> &mdash; it is deliberately cycling
through every arm, including the bad ones, so its first ~10 steps are close to random.
<br><br><b>Late:</b> optimistic greedy wins clearly. It stops exploring once the estimates
settle and then exploits at nearly 100%, while &epsilon;-greedy is stuck under its
1-&epsilon;+&epsilon;/k = 91% ceiling forever.
<br><br><b>Shape:</b> not smooth &mdash; there is a pronounced <b>spike around step 11</b>
and often a damped oscillation with period ~10 afterwards. That is Exercise 2.6, worked
through below.''')

## Exercise 2.6 — Mysterious Spikes

> The results shown in Figure 2.3 should be quite reliable because they are averages over
> 2000 individual, randomly chosen 10-armed bandit tasks. Why, then, are there
> oscillations and spikes in the early part of the curve for the optimistic method? In
> other words, what might make this method perform particularly better or worse, on
> average, on particular early steps?

The key: **the optimistic agent's behaviour in the first 10 steps is nearly deterministic
in structure, and identical across runs.** It samples all 10 arms in some order (ties
broken randomly, so the *order* differs, but the *pattern* does not).

- **Steps 1–10:** each arm is visited roughly once. The chance of hitting the optimal arm
  on any given one of these is about $1/10$, so % optimal hovers near 10%.
- **Step 11:** every arm has now been sampled once and knocked down from 5 by roughly
  $\alpha(5 - q_*(a) - \text{noise})$. The agent now picks the arm with the highest single
  sample — which is the optimal arm far more often than chance. **Spike.**
- **Steps 12–20:** that arm gets sampled again, gets knocked down again below the others,
  and the agent goes back to sweeping. **Dip.**
- The synchronisation across runs decays as the visit counts desynchronise, so the
  oscillation damps out.

The spike is an artefact of *phase alignment across the 2000 runs*, not of any run doing
something special. Let us verify by measuring the sweep directly.

In [ ]:
def optimistic_trace(steps=40, runs=3000, Q_init=5.0, alpha=0.1, seed=0):
    tb = Testbed(runs=runs, seed=seed)
    rng = np.random.default_rng(seed + 99)
    Q = np.full((runs, 10), float(Q_init))
    distinct, pct_opt, is_new = [], [], []
    seen = np.zeros((runs, 10), bool)
    best = tb.optimal()
    for t in range(steps):
        a = argmax_random_tiebreak(Q, rng)
        idx = np.arange(runs)
        is_new.append((~seen[idx, a]).mean())
        seen[idx, a] = True
        r = tb.step(a)
        Q[idx, a] += alpha * (r - Q[idx, a])
        distinct.append(seen.sum(axis=1).mean())
        pct_opt.append((a == best).mean() * 100)
    return np.array(distinct), np.array(pct_opt), np.array(is_new)

distinct, pct_opt, is_new = optimistic_trace()

fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
axes[0].plot(np.arange(1, 41), pct_opt, "o-", color="crimson")
axes[0].axhline(10, ls=":", color="gray")
axes[0].set_ylabel("% optimal")
axes[0].set_title("The spike at step 11 and the damped echo near 21")

axes[1].plot(np.arange(1, 41), distinct, "o-")
axes[1].plot(np.arange(1, 41), np.minimum(np.arange(1, 41), 10), ls="--",
             color="gray", label="if it swept perfectly")
axes[1].set_ylabel("distinct arms tried")
axes[1].legend(fontsize=8)

axes[2].plot(np.arange(1, 41), is_new, "o-", color="darkgreen")
axes[2].set_ylabel("P(action is brand new)")
axes[2].set_xlabel("Step")
for ax in axes:
    ax.axvline(11, color="gray", ls="--", lw=1); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("distinct arms tried by step 10:", round(distinct[9], 2), "/ 10")
print("%opt at step 10, 11, 12:", np.round(pct_opt[9:12], 1))

The middle panel confirms it: by step 10 the agent has tried **all 10** arms — a
perfect systematic sweep, driven entirely by greedy action selection with no randomness
in the decision rule at all. The bottom
panel shows the probability of trying something new collapsing right at step 11, which
is precisely where the spike lands.

## When optimism breaks: the nonstationary case

The whole mechanism fires once, at the start. Put it on a drifting problem and watch.

---

### Predict first

On the nonstationary testbed from 2.5 (drifting $q_*$, 10,000 steps), how
will optimistic greedy compare to plain $\varepsilon$-greedy with constant $\alpha$?
Will optimism help, hurt, or be roughly neutral?

*Write your guess down (mentally or in the cell below) before running the next cell. The point is not to be right — it is to make the surprise informative when you are wrong.*

In [ ]:
kw = {"drift_sigma": 0.01, "equal_start": True}
T = 10000
opt_ns = run_bandit(eps_agent(0.0, Q_init=5.0, alpha=0.1), testbed_kwargs=kw,
                    steps=T, seed=3)
eps_ns = run_bandit(eps_agent(0.1, Q_init=0.0, alpha=0.1), testbed_kwargs=kw,
                    steps=T, seed=3)
opt_eps_ns = run_bandit(eps_agent(0.1, Q_init=5.0, alpha=0.1), testbed_kwargs=kw,
                        steps=T, seed=3)

plot_pair([opt_ns, eps_ns, opt_eps_ns],
          ["optimistic greedy ($\\varepsilon$=0)",
           "realistic $\\varepsilon$=0.1",
           "optimistic + $\\varepsilon$=0.1"],
          title="Optimism on a nonstationary problem")
plt.show()

for lab, r in [("optimistic greedy", opt_ns), ("eps-greedy", eps_ns),
               ("both", opt_eps_ns)]:
    print(f"{lab:<20} %opt last 2000 = {r['optimal'][-2000:].mean():5.1f}   "
          f"reward last 2000 = {r['rewards'][-2000:].mean():.3f}")

In [ ]:
hide('''Optimistic greedy starts fine and then <b>rots</b>. Once its estimates have
deflated to reality it is pure greedy, and pure greedy on a drifting problem locks onto
an arm that was good ten thousand steps ago. &epsilon;-greedy, which keeps sampling
forever, ends up well ahead.
<br><br>Notice the third curve: optimism + &epsilon; is essentially just &epsilon;-greedy
after the first hundred steps. The optimistic initialisation contributes a better start
and nothing after that.
<br><br>This is the core critique in the book: optimistic initial values are a
<i>one-shot</i> mechanism attached to the beginning of time. Any problem where the world
keeps changing needs exploration that is driven by <i>ongoing uncertainty</i>, not by the
calendar. That is exactly what UCB provides.''')

### Interactive: how optimistic is optimistic enough?

In [ ]:
try:
    import ipywidgets as W
    HAVE_WIDGETS = True
except ImportError:
    HAVE_WIDGETS = False

def optimism_sweep(Q_init=5.0, alpha=0.1, eps=0.0):
    r = run_bandit(eps_agent(eps, Q_init=Q_init, alpha=alpha), runs=400, seed=5)
    base = run_bandit(eps_agent(0.1, Q_init=0.0, alpha=0.1), runs=400, seed=5)
    plot_pair([base, r], ["baseline $Q_1{=}0,\\varepsilon{=}0.1$",
                          f"$Q_1$={Q_init}, $\\alpha$={alpha}, $\\varepsilon$={eps}"])
    plt.show()

if HAVE_WIDGETS:
    W.interact(optimism_sweep,
               Q_init=W.FloatSlider(value=5, min=-2, max=20, step=1),
               alpha=W.FloatLogSlider(value=0.1, base=10, min=-2, max=-0.15, step=0.1),
               eps=W.FloatSlider(value=0.0, min=0.0, max=0.3, step=0.02))
else:
    for q in [0, 5, 20]:
        optimism_sweep(Q_init=q)

Things worth discovering with that slider:

- Very large $Q_1$ (say 20) makes the initial sweep last much longer — the agent has to
  knock every arm down from 20 before it settles. Better asymptotics, worse early reward.
- Small $\alpha$ makes optimism *stickier* (the $(1-\alpha)^n$ term decays slower), so the
  exploratory phase stretches out.
- With sample-average updates instead of constant $\alpha$, optimism is much weaker: the
  first pull of each arm wipes $Q_1$ out entirely.

## Takeaways for 2.6

1. Optimistic initialisation makes a pure-greedy agent explore, because every untried
   action looks best.
2. It gives near-100% asymptotic optimal-action rate — no $1-\varepsilon+\varepsilon/k$
   ceiling.
3. The Figure 2.3 spike at step 11 comes from the systematic sweep of all 10 arms being
   phase-aligned across runs.
4. It is a transient tied to $t=0$: useless for nonstationary problems, and it presumes
   you know the reward scale.

**Next:** exploration driven by *how uncertain you currently are*, not by when you
started — Section 2.7.